In [42]:
# Imports
!pip install -q hopsworks pandas requests numpy matplotlib seaborn confluent-kafka xgboost tensorflow shap
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import hopsworks
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

CITY = "Lahore"
DAYS_TO_FETCH    = 365
HOPSWORKS_API_KEY = userdata.get("hopscotch")

In [43]:
# fetching air quality data from open meteo
def get_city_coordinates(city_name):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    response = requests.get(url)
    data = response.json()
    if "results" in data:
        return data["results"][0]["latitude"], data["results"][0]["longitude"], data["results"][0]["name"]
    else:
        raise ValueError(f"City '{city_name}' not found.")

lat, lon, city = get_city_coordinates(CITY)
print(f"Coordinates for {city}: Lat={lat}, Lon={lon}")

def fetch_aqi_data_openmeteo(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    # meteo's api endpoint
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust",
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "timezone": "auto"
    }

    print(f"Air Quality data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"Open-Meteo AQI API Error: {data}")

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "us_aqi": data["hourly"]["us_aqi"],
        "pm25": data["hourly"]["pm2_5"],
        "pm10": data["hourly"]["pm10"],
        "co": data["hourly"]["carbon_monoxide"],
        "no2": data["hourly"]["nitrogen_dioxide"],
        "so2": data["hourly"]["sulphur_dioxide"],
        "o3": data["hourly"]["ozone"],
        "dust": data["hourly"]["dust"]
    })

    return df

df_aqi = fetch_aqi_data_openmeteo(lat, lon, DAYS_TO_FETCH)
print(f"Fetched {len(df_aqi)} records.")
display(df_aqi.head(10))

Coordinates for Lahore: Lat=31.558, Lon=74.35071
Air Quality data from 2025-07-30 to 2026-07-30...
Fetched 8784 records.


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust
0,2025-07-30 00:00:00,130,47.1,126.2,491.0,17.7,8.4,92.0,141.0
1,2025-07-30 01:00:00,127,53.1,165.3,416.0,16.7,8.3,84.0,205.0
2,2025-07-30 02:00:00,124,58.7,190.9,363.0,16.2,8.3,76.0,256.0
3,2025-07-30 03:00:00,123,61.5,199.8,351.0,16.2,8.4,69.0,279.0
4,2025-07-30 04:00:00,123,65.0,208.2,362.0,16.5,8.6,63.0,288.0
5,2025-07-30 05:00:00,124,67.8,220.7,386.0,16.9,9.1,61.0,295.0
6,2025-07-30 06:00:00,130,69.2,221.7,436.0,17.7,10.2,63.0,311.0
7,2025-07-30 07:00:00,135,71.0,254.3,500.0,18.5,11.6,68.0,324.0
8,2025-07-30 08:00:00,141,69.2,228.6,532.0,17.9,12.4,80.0,309.0
9,2025-07-30 09:00:00,146,59.4,167.2,507.0,14.6,12.2,105.0,241.0


In [44]:
# getting weather data for accurate prediction of next days as current air quality not enough to predict future.
def fetch_weather_data(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }

    print(f"Weather data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "temperature_c": data["hourly"]["temperature_2m"],
        "humidity_pct": data["hourly"]["relative_humidity_2m"],
        "wind_speed_kmh": data["hourly"]["wind_speed_10m"],
        "precipitation_mm": data["hourly"]["precipitation"]
    })

    return df

df_weather = fetch_weather_data(lat, lon, DAYS_TO_FETCH)

# merge aqi and weather df for 1 single df
df_raw = pd.merge(df_aqi, df_weather, on="timestamp", how="inner")
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print(f"Final raw dataset shape: {df_raw.shape}")
display(df_raw.head(10))

Weather data from 2025-07-30 to 2026-07-30...
Final raw dataset shape: (8784, 13)


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,humidity_pct,wind_speed_kmh,precipitation_mm
0,2025-07-30 00:00:00,130,47.1,126.2,491.0,17.7,8.4,92.0,141.0,26.5,96,10.9,1.0
1,2025-07-30 01:00:00,127,53.1,165.3,416.0,16.7,8.3,84.0,205.0,26.9,97,6.6,0.4
2,2025-07-30 02:00:00,124,58.7,190.9,363.0,16.2,8.3,76.0,256.0,26.7,97,12.8,1.5
3,2025-07-30 03:00:00,123,61.5,199.8,351.0,16.2,8.4,69.0,279.0,26.5,95,12.0,1.1
4,2025-07-30 04:00:00,123,65.0,208.2,362.0,16.5,8.6,63.0,288.0,26.2,94,14.9,2.9
5,2025-07-30 05:00:00,124,67.8,220.7,386.0,16.9,9.1,61.0,295.0,26.0,95,13.8,0.4
6,2025-07-30 06:00:00,130,69.2,221.7,436.0,17.7,10.2,63.0,311.0,26.3,95,15.7,0.1
7,2025-07-30 07:00:00,135,71.0,254.3,500.0,18.5,11.6,68.0,324.0,27.2,91,14.0,0.0
8,2025-07-30 08:00:00,141,69.2,228.6,532.0,17.9,12.4,80.0,309.0,28.1,88,12.8,0.1
9,2025-07-30 09:00:00,146,59.4,167.2,507.0,14.6,12.2,105.0,241.0,29.4,81,13.3,0.1


In [45]:
# feature engineerinh
# get features from timestamp to get periodic patterns
df_raw["hour"] = df_raw["timestamp"].dt.hour
df_raw["day"] = df_raw["timestamp"].dt.day
df_raw["month"] = df_raw["timestamp"].dt.month
df_raw["dayofweek"] = df_raw["timestamp"].dt.dayofweek  # 0=Monday - 6=Sunday

# reading of aqi and pm25 for an hour and day ago repesented by lag (lagging)
# df_raw["aqi_lag_1"] = df_raw["us_aqi"].shift(1)
# df_raw["aqi_lag_24"] = df_raw["us_aqi"].shift(24)
df_raw["pm25_lag_1"] = df_raw["pm25"].shift(1)
df_raw["pm25_lag_1"] = df_raw["pm25"].shift(24)

# how much aqi changed from previous hour
# df_raw["aqi_change_rate"] = df_raw["us_aqi"] - df_raw["aqi_lag_1"]

# rolling average
df_raw["pm25_rolling_24h"] = df_raw["pm25"].rolling(window=24, min_periods=1).mean()

# drop rows with nan entries for clean dataset
df_features = df_raw.dropna().reset_index(drop=True)

print(f"Original rows: {len(df_raw)} | Final  rows: {len(df_features)}")
print("New Dataset Columns:")
print(df_features.columns.tolist())
display(df_features.head(10))

Original rows: 8784 | Final  rows: 8760
New Dataset Columns:
['timestamp', 'us_aqi', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3', 'dust', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'hour', 'day', 'month', 'dayofweek', 'pm25_lag_1', 'pm25_rolling_24h']


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,humidity_pct,wind_speed_kmh,precipitation_mm,hour,day,month,dayofweek,pm25_lag_1,pm25_rolling_24h
0,2025-07-31 00:00:00,141,59.8,59.8,618.0,43.3,11.7,41.0,0.0,27.7,90,8.9,0.0,0,31,7,3,47.1,52.291667
1,2025-07-31 01:00:00,142,57.2,57.2,509.0,36.5,11.4,43.0,0.0,27.5,91,8.1,0.0,1,31,7,3,53.1,52.462500
2,2025-07-31 02:00:00,142,54.2,54.2,435.0,31.7,11.0,42.0,0.0,27.2,92,7.3,0.0,2,31,7,3,58.7,52.275000
3,2025-07-31 03:00:00,142,52.2,52.3,432.0,31.1,10.4,35.0,0.0,26.9,91,5.9,0.0,3,31,7,3,61.5,51.887500
4,2025-07-31 04:00:00,141,51.3,51.6,465.0,32.6,9.6,27.0,0.0,26.6,90,6.6,0.0,4,31,7,3,65.0,51.316667
5,2025-07-31 05:00:00,140,48.3,50.2,499.0,27.8,11.0,33.0,4.0,26.3,91,6.2,0.0,5,31,7,3,67.8,50.504167
6,2025-07-31 06:00:00,138,46.7,49.2,534.0,26.2,12.7,43.0,12.0,27.2,88,7.8,0.0,6,31,7,3,69.2,49.566667
7,2025-07-31 07:00:00,135,44.1,50.9,569.0,23.8,15.0,58.0,22.0,28.0,88,5.1,0.0,7,31,7,3,71.0,48.445833
8,2025-07-31 08:00:00,132,41.7,55.5,569.0,20.8,15.8,72.0,27.0,29.0,83,11.5,0.0,8,31,7,3,69.2,47.300000
9,2025-07-31 09:00:00,130,35.5,51.5,498.0,16.7,13.6,83.0,22.0,29.4,80,12.9,0.0,9,31,7,3,59.4,46.304167


In [46]:
project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY)

fs = project.get_feature_store()

feature_group = fs.get_or_create_feature_group(
    name="aqi_features",             # table name
    version=3,
    description="Hourly AQI, weather, and engineered features for ML training",
    primary_key=["timestamp"],
    event_time="timestamp",           # tells hopsworks this is time-series data
    time_travel_format="HUDI"
)

insert_response = feature_group.insert(df_features)


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42146
Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42146/fs/30836/fg/51458


Uploading Dataframe: 100.00% |██████████| Rows 8760/8760 | Elapsed Time: 00:07 | Remaining Time: 00:00


Launching job: aqi_features_3_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/42146/jobs/named/aqi_features_3_offline_fg_materialization/executions


In [52]:
# project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY)
# fs = project.get_feature_store()

# fg = fs.get_feature_group(name="aqi_features", version=2)

df_train = df_features
print(f"Fetched {len(df_train)} rows for training.")

# dropping time stamp and aqi
X = df_train.drop(columns=["timestamp", "us_aqi"])
y = df_train["us_aqi"]

# 80:20 split
split_idx = len(X) - 3
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set size: {len(X_train)} | Test set size: {len(X_test)}")




Fetched 8760 rows for training.
Training set size: 8757 | Test set size: 3


In [53]:

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"{model_name} Evaluation:")
    print(f"   RMSE: {rmse:.2f}")
    print(f"   MAE:  {mae:.2f}")
    print(f"   R²:   {r2:.4f}")
    return [rmse, r2, mae] , model

In [54]:
print("Training models...")
# ridge and random forest as stated in writeup

# model 1: ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_eval, ridge_best = evaluate_model(ridge_model, X_test, y_test, "Ridge Regression")

# model 2: random forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_eval, rf_best = evaluate_model(rf_model, X_test, y_test, "Random Forest")

# xgboost for a furthur

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_eval, xgb_best = evaluate_model(xgb_model, X_test, y_test, "3. XGBoost")



Training models...
Ridge Regression Evaluation:
   RMSE: 30.31
   MAE:  26.32
   R²:   -2.2264
Random Forest Evaluation:
   RMSE: 11.21
   MAE:  9.68
   R²:   0.5588
3. XGBoost Evaluation:
   RMSE: 14.45
   MAE:  10.88
   R²:   0.2668


In [55]:
print("Training deep learning models...")

# first a mlp, then lstm for sequential data, and gru as its simpler version of lstm
tf_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
tf_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = tf_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)
tf_eval, tf_best = evaluate_model(tf_model, X_test, y_test, "4. MLP")

# lstm and gru data preprocessin
X_rnn = df_train.drop(columns=["timestamp", "us_aqi"])
y_rnn = df_train["us_aqi"]

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Creating sequence for lstm and gru to learn
TIME_STEPS = 24
X_seq, y_seq = [], []

for i in range(len(X_scaled) - TIME_STEPS):
    X_seq.append(X_scaled[i : i + TIME_STEPS])
    y_seq.append(y_rnn.iloc[i + TIME_STEPS])

X_seq, y_seq = np.array(X_seq), np.array(y_seq)
print(f"Created {len(X_seq)} sequences. 3D Shape: {X_seq.shape}")

split_idx = len(X_seq) - 3
X_train_seq, X_test_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_test_seq = y_seq[:split_idx], y_seq[split_idx:]


# Early stopping to prevent overfitting (very common in LSTMs)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
print("Training LSTM Model...")
lstm_model = Sequential([
    LSTM(64, activation='relu', input_shape=(TIME_STEPS, X_seq.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train_seq, y_train_seq, validation_split=0.2, epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
lstm_eval, lstm_best = evaluate_model(lstm_model, X_test_seq, y_test_seq, "5. LSTM")

print("Training GRU Model...")
gru_model = Sequential([
    GRU(64, activation='relu', input_shape=(TIME_STEPS, X_seq.shape[2]), return_sequences=True),
    Dropout(0.2),
    GRU(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1) # Output layer
])
gru_model.compile(optimizer='adam', loss='mse')

gru_model.fit(X_train_seq, y_train_seq, validation_split=0.2, epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
gru_eval, gru_best = evaluate_model(gru_model, X_test_seq, y_test_seq, "6. GRU")



Training deep learning models...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
4. MLP Evaluation:
   RMSE: 22.01
   MAE:  16.73
   R²:   -0.7021
Created 8736 sequences. 3D Shape: (8736, 24, 17)
Training LSTM Model...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step
5. LSTM Evaluation:
   RMSE: 8.86
   MAE:  7.92
   R²:   0.7240
Training GRU Model...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
6. GRU Evaluation:
   RMSE: 32.63
   MAE:  29.21
   R²:   -2.7404


In [56]:
model_results = {
    "Ridge Regression": ridge_eval,
    "Random Forest": rf_eval,
    "XGBoost": xgb_eval,
    "MLP (Deep Learning)": tf_eval,
    "LSTM": lstm_eval,
    "GRU": gru_eval
}

df_results = pd.DataFrame.from_dict(
    model_results,
    orient="index",
    columns=["RMSE", "R²", "MAE"]
)

# sort the grid by the best metric
df_results = df_results.sort_values(by="RMSE", ascending=True)

df_results["RMSE"] = df_results["RMSE"].map("{:.2f}".format)
df_results["MAE"] = df_results["MAE"].map("{:.2f}".format)
df_results["R²"] = df_results["R²"].map("{:.4f}".format)

print("MODEL PERFORMANCE LEADERBOARD (Sorted by lowest RMSE)")
display(df_results)


MODEL PERFORMANCE LEADERBOARD (Sorted by lowest RMSE)


,RMSE,R²,MAE
LSTM,8.86,0.7240,7.92
Random Forest,11.21,0.5588,9.68
XGBoost,14.45,0.2668,10.88
MLP (Deep Learning),22.01,-0.7021,16.73
Ridge Regression,30.31,-2.2264,26.32
GRU,32.63,-2.7404,29.21
